In [ ]:
# 1. Preparação
from pathlib import Path
import pandas as pd

arquivo = Path('Base_Escola.csv')
saida = Path('resultados')
data_teste = pd.Timestamp('2026-11-02')
multa_teste = 0.02  # valor usado somente nos testes
frequencia_minima = 75  # valor usado somente nos testes

assert arquivo.exists(), 'Envie Base_Escola.csv pela pasta do Colab antes de continuar.'


In [ ]:
# 2. Leitura dos dados
original = pd.read_csv(arquivo, dtype=str, keep_default_na=False, encoding='utf-8-sig')
colunas = [
    'id_aluno', 'nome', 'modalidade', 'turma', 'faixa_atual',
    'valor_mensalidade', 'vencimento', 'data_pagamento',
    'quantidade_aulas', 'presencas'
]
assert set(colunas).issubset(original.columns), 'O CSV está sem alguma coluna necessária.'

print('Linhas da base:', len(original))
display(original.head())


In [ ]:
# 3. Padronização
modalidades = {
    'jiu-jitsu': 'Jiu-jitsu', 'jiu jitsu': 'Jiu-jitsu',
    'muay thai': 'Muay Thai', 'judô': 'Judô', 'judo': 'Judô',
    'karatê': 'Karatê', 'karate': 'Karatê',
}

dados = original[colunas].copy()
dados['linha'] = dados.index + 2  # a linha 1 do CSV é o cabeçalho
dados['id_aluno'] = dados['id_aluno'].str.strip()
dados['nome'] = dados['nome'].str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
dados['modalidade'] = (dados['modalidade'].str.strip()
                       .str.replace(r'\s+', ' ', regex=True)
                       .str.lower().map(modalidades))
dados['turma'] = dados['turma'].str.strip().str.upper()
dados['faixa_atual'] = (dados['faixa_atual'].str.strip()
                        .str.replace(r'\s+', ' ', regex=True).str.title())

print('Campos de texto padronizados.')
display(dados[['nome', 'modalidade', 'turma', 'faixa_atual']].head())


In [ ]:
# 4. Valores e datas
texto_valor = (dados['valor_mensalidade'].str.strip()
              .str.replace('R$', '', regex=False)
              .str.replace(' ', '', regex=False)
              .str.replace(',', '.', regex=False))
valor = pd.to_numeric(texto_valor, errors='coerce')
texto_aulas = dados['quantidade_aulas'].str.strip()
texto_presencas = dados['presencas'].str.strip()
aulas = pd.to_numeric(texto_aulas, errors='coerce')
presencas = pd.to_numeric(texto_presencas, errors='coerce')

def ler_datas(serie):
    formato_ano = pd.to_datetime(serie, format='%Y-%m-%d', errors='coerce')
    formato_br = pd.to_datetime(serie, format='%d/%m/%Y', errors='coerce')
    return formato_ano.fillna(formato_br)

vencimento = ler_datas(dados['vencimento'])
texto_pagamento = dados['data_pagamento'].str.strip()
pagamento = ler_datas(texto_pagamento)

print('Valores e datas convertidos.')


In [ ]:
# 5. Validação
motivos = pd.Series('', index=dados.index, dtype='str')

def marcar_erro(condicao, motivo):
    motivos.loc[motivos.eq('') & condicao.fillna(False)] = motivo

marcar_erro(dados['id_aluno'].eq('') | dados['nome'].eq('') |
            dados['turma'].eq('') | dados['faixa_atual'].eq(''),
            'cadastro com campo obrigatório vazio')
marcar_erro(dados['modalidade'].isna(), 'modalidade não reconhecida')
marcar_erro(valor.isna() | aulas.isna() | presencas.isna() |
            ~texto_aulas.str.fullmatch(r'[+-]?\d+') |
            ~texto_presencas.str.fullmatch(r'[+-]?\d+'), 'número inválido')
marcar_erro(valor.le(0), 'valor da mensalidade inválido')
marcar_erro(aulas.le(0) | presencas.lt(0) | presencas.gt(aulas),
            'quantidade de aulas ou presenças inválida')
marcar_erro(vencimento.isna() | (texto_pagamento.ne('') & pagamento.isna()),
            'data de vencimento ou pagamento inválida')
marcar_erro(pagamento.gt(data_teste), 'pagamento com data futura')

# Confere os IDs repetidos
candidatos = dados.loc[motivos.eq('')]
repetidos = candidatos.duplicated(subset='id_aluno', keep='first')
motivos.loc[candidatos.index[repetidos]] = 'identificador repetido'

validos = motivos.eq('')
rejeitados = pd.DataFrame({
    'linha': dados.loc[~validos, 'linha'],
    'id_aluno': original.loc[~validos, 'id_aluno'],
    'motivo': motivos.loc[~validos]
})

print('Registros aproveitados:', int(validos.sum()))
print('Registros rejeitados:', len(rejeitados))
display(rejeitados.head(8))


In [ ]:
# 6. Pagamentos e frequência
tratados = dados.loc[validos, colunas].copy()
tratados['valor_mensalidade'] = valor.loc[validos].map(lambda n: f'{n:.2f}')
tratados['vencimento'] = vencimento.loc[validos].dt.strftime('%Y-%m-%d')
tratados['data_pagamento'] = pagamento.loc[validos].dt.strftime('%Y-%m-%d').fillna('')
tratados['quantidade_aulas'] = aulas.loc[validos].astype(int)
tratados['presencas'] = presencas.loc[validos].astype(int)

tratados['situacao_pagamento'] = 'Em aberto'
tratados.loc[vencimento.loc[validos].lt(data_teste), 'situacao_pagamento'] = 'Atrasado'
tratados.loc[pagamento.loc[validos].notna(), 'situacao_pagamento'] = 'Pago'
tratados['multa_exemplo'] = '0.00'
atrasados = tratados['situacao_pagamento'].eq('Atrasado')
tratados.loc[atrasados, 'multa_exemplo'] = (
    valor.loc[tratados.index[atrasados]] * multa_teste
).map(lambda n: f'{n:.2f}')

tratados['frequencia_percentual'] = (
    100 * tratados['presencas'] / tratados['quantidade_aulas']
).round(1)
tratados['frequencia_minima_atingida'] = (
    tratados['frequencia_percentual'] >= frequencia_minima
).map({True: 'Sim', False: 'Não'})

display(tratados.head(6))


In [ ]:
# 7. Correções
lista_correcoes = []
campos_conferidos = [
    'nome', 'modalidade', 'turma', 'faixa_atual',
    'valor_mensalidade', 'vencimento', 'data_pagamento'
]

for posicao, campo in enumerate(campos_conferidos):
    alterou = original.loc[validos, campo].ne(tratados[campo])
    indices = tratados.index[alterou]
    if len(indices) > 0:
        lista_correcoes.append(pd.DataFrame({
            'linha': indices + 2,
            'id_aluno': tratados.loc[indices, 'id_aluno'].to_numpy(),
            'campo': campo,
            'antes': original.loc[indices, campo].to_numpy(),
            'depois': tratados.loc[indices, campo].to_numpy(),
            '_ordem': posicao,
        }))

if lista_correcoes:
    correcoes = (pd.concat(lista_correcoes, ignore_index=True)
                .sort_values(['linha', '_ordem']).drop(columns='_ordem'))
else:
    correcoes = pd.DataFrame(columns=['linha', 'id_aluno', 'campo', 'antes', 'depois'])

print('Registros que tiveram correção:', correcoes['linha'].nunique())
print('Campos corrigidos:', len(correcoes))
display(correcoes.head(10))


In [ ]:
# 8. Arquivos de saída
saida.mkdir(exist_ok=True)
tratados.to_csv(saida / 'dados_tratados.csv', index=False, encoding='utf-8-sig')
correcoes.to_csv(saida / 'correcoes.csv', index=False, encoding='utf-8-sig')
rejeitados.to_csv(saida / 'rejeitados.csv', index=False, encoding='utf-8-sig')

situacoes = tratados['situacao_pagamento'].value_counts()
resumo = (f'TRATAMENTO DA BASE DA ESCOLA - DADOS FICTÍCIOS\n'
          f'Biblioteca: Pandas\n'
          f'Data do teste: {data_teste.date()}\n'
          f'Registros lidos: {len(original)}\n'
          f'Registros aproveitados: {len(tratados)}\n'
          f'Registros com algum campo corrigido: {correcoes["linha"].nunique()}\n'
          f'Correções de campos: {len(correcoes)}\n'
          f'Registros rejeitados: {len(rejeitados)}\n'
          f'Mensalidades pagas: {situacoes.get("Pago", 0)}\n'
          f'Mensalidades em aberto: {situacoes.get("Em aberto", 0)}\n'
          f'Mensalidades atrasadas: {situacoes.get("Atrasado", 0)}\n'
          'Multa e frequência mínima são regras provisórias usadas apenas no teste.\n')
(saida / 'resumo.txt').write_text(resumo, encoding='utf-8')
print(resumo)
print('Arquivos salvos em:', saida)
